# COMP47470 Assignment 2: MongoDB

This assignment is about MongoDB.

**You can complete this assignment using MongoDB on your local machine and just include the solutions in the code or text cells.**

**To install and quickstart MongoDB on your local machine, follow the instructions provided on Brightspace in the folder About the tools.**

**You are also allowed to use PyMongo to submit the solutions in Python.**

The assignment carries 10 points (10% weight). It has two components.

Before you solve the questions, you must prepare a MongoDB database using the instructions below.

## MongoDB Component 1

You are given a financial transactions database, in a file **comp47470_financial_transactions.json**.

The database contains detailed financial transactions. This collection contains transactions details for users. Each document contains an account id, a count of how many transactions are in this set, the start and end dates for transactions covered by this document, and a list of sub documents. Each sub document represents a single transaction and the related information for that transaction.

You can read the format of a document from the URL below. It will assist you to identify the keys of a document.
https://www.mongodb.com/docs/atlas/sample-data/sample-analytics/#std-label-analytics-transactions

```json
{
  "account_id": 794875,
  "transaction_count": 6,
  "bucket_start_date": {"$date": 693792000000},
  "bucket_end_date": {"$date": 1473120000000},
  "transactions": [
    {
      "date": {"$date": 1325030400000},
      "amount": 1197,
      "transaction_code": "buy",
...
```

Follow the instructions below to load the weatherdata collection into MongoDB.

You will need to complete this step successfully to proceed to answer the questions in the component.

1). Start your MongoDB daemon, **mongod**.

2). Import the weatherdata collection using the command invocation below:

**mongoimport --db=comp47470financialdb --collection=transactions --file=comp47470_financial_transactions.json**

On successful invocation of the command, you will get the following output:

```text
2025-09-16T18:40:46.756+0100	connected to: mongodb://localhost/
2025-09-16T18:40:46.990+0100	1746 document(s) imported successfully. 0 document(s) failed to import.
```

Launch the MongoDB shell, **mongosh**.

At the command prompt, type 'show databases'. You will see 'comp47470weatherdata' db listed.

```bash
test> show databases
admin                  40.00 KiB
comp47470financialdb    9.20 MiB
config                 72.00 KiB
local                  72.00 KiB
test                  112.00 KiB
```

Switch to 'comp47470financialdb' and find which collections are available in the database.

```bash
test> use comp47470financialdb
switched to db comp47470financialdb
comp47470weatherdata> show collections
transactions
```

Run the following queries:

1). Select one document from the collection **transactions** using the cursor **limit** directive as follows:

2). Find the number of documents. Use **countDocuments()** since it provides the exact count of documents.

```bash
comp47470financialdb> db.transactions.find( {} ).limit(1)
[
  {
    _id: ObjectId('5ca4bbc1a2dd94ee58161cb1'),
    account_id: 443178,
    transaction_count: 66,
    bucket_start_date: ISODate('1969-02-04T00:00:00.000Z'),
    bucket_end_date: ISODate('2017-01-03T00:00:00.000Z'),
    transactions: [
      {
        date: ISODate('2003-09-09T00:00:00.000Z'),
...

comp47470financialdb> db.transactions.countDocuments( {} )
1746

```

If you are able to run this query successfully, then you are ready to tackle the following questions in the component.

In [ ]:
from pymongo import MongoClient
import json
from bson import json_util

client = MongoClient("mongodb://localhost:27017/")
db = client["comp47470financialdb"]
collection = db["transactions"]


json_file_path = "comp47470_financial_transactions.json"
with open(json_file_path, 'r') as f:
    data = [json_util.loads(line) for line in f]


collection.insert_many(data)
print(f"{len(data)} documents inserted successfully!")


1746 documents inserted successfully!


## Problem 1

Write a MongoDB aggregation pipeline that outputs the **total quantity** and **total price** of transactions involving Amazon?

For more on aggregation pipelines, follow the URL below:

https://www.mongodb.com/docs/manual/core/aggregation-pipeline/

Note the field **symbol** with the value 'amzn' denotes transactions involving Amazon. The nested field **transactions.amount** gives the quantity for each transaction. The nested field **transactions.total** gives the total price for each transaction.

The group stage will calculate the total quantity, which will be stored in **totalqty** and the total price, which will be stored in **totalprice**.

**The total price must be rounded to two decimal places.**

The output must contain only the following:

1.   Symbol
2.   Total Quantity
3.   Total Price

**What is the statement and the output from the statement execution? (2 points)**?

In [ ]:
from bson.son import SON

pipeline = [
    
    {"$unwind": "$transactions"},
    
    
    {"$match": {"transactions.symbol": "amzn"}},
    
    
    {"$group": {
        "_id": "$transactions.symbol",
        "totalqty": {"$sum": "$transactions.amount"},
        "totalprice": {"$sum": "$transactions.total"}
    }},
    
    
    {"$project": {
        "Symbol": "$_id",
        "Total Quantity": "$totalqty",
        "Total Price": {"$round": ["$totalprice", 2]},
        "_id": 0
    }}
]


result = list(collection.aggregate(pipeline))


from pprint import pprint
pprint(result)


[{'Symbol': 'amzn', 'Total Price': 0, 'Total Quantity': 25511429}]


## Problem 2

Write a MongoDB aggregation pipeline that outputs the difference between the **total price** of **sales** and **purchases** involving Ebay?

Note the field **symbol** with the value 'ebay' denotes transactions involving Ebay. The nested field **transactions.transaction_code** gives the type of transaction, 'buy' or 'sell. The nested field **transactions.total** gives the total price for each transaction.

The group stage will calculate the total price of sales, which will be stored in **totalsell** and the total price of purchases in **totalbuy**. You need to find the difference between the two.

The output must contain only the following:

1.   Symbol
2.   Total price for sales
3.   Total Price for purchases
4.   Difference between 2 and 3

**All the prices must be rounded to two decimal places.**

**What is the statement and the output from the statement execution? (3 points)**?

In [ ]:

pipeline_check = [
    {"$unwind": "$transactions"},
    {"$group": {"_id": "$transactions.symbol"}},
    {"$sort": {"_id": 1}}
]

symbols = list(collection.aggregate(pipeline_check))
symbols


[{'_id': 'aapl'},
 {'_id': 'adbe'},
 {'_id': 'amd'},
 {'_id': 'amzn'},
 {'_id': 'bb'},
 {'_id': 'crm'},
 {'_id': 'csco'},
 {'_id': 'ebay'},
 {'_id': 'fb'},
 {'_id': 'goog'},
 {'_id': 'ibm'},
 {'_id': 'intc'},
 {'_id': 'msft'},
 {'_id': 'nflx'},
 {'_id': 'nvda'},
 {'_id': 'sap'},
 {'_id': 'team'},
 {'_id': 'znga'}]

In [ ]:
pipeline = [
    {"$unwind": "$transactions"}, 
    {"$match": {"transactions.symbol": "ebay"}},  
    {"$group": {
        "_id": "$transactions.symbol",
        "Total_Sales": {"$sum": {
            "$cond": [{"$eq": ["$transactions.transaction_code", "sell"]}, 
                      {"$toDouble": "$transactions.total"}, 0]
        }},
        "Total_Purchases": {"$sum": {
            "$cond": [{"$eq": ["$transactions.transaction_code", "buy"]}, 
                      {"$toDouble": "$transactions.total"}, 0]
        }}
    }},
    {"$project": {
        "Symbol": "$_id",
        "Total Sales": {"$round": ["$Total_Sales", 2]},
        "Total Purchases": {"$round": ["$Total_Purchases", 2]},
        "Difference": {"$round": [{"$subtract": ["$Total_Sales", "$Total_Purchases"]}, 2]},
        "_id": 0
    }}
]

result = list(collection.aggregate(pipeline))
result

[{'Symbol': 'ebay',
  'Total Sales': 223968043.8,
  'Total Purchases': 220359449.85,
  'Difference': 3608593.96}]

## MongoDB Component 2

You are given a office supplies database, in a file **supplies.json**.

The database contains detailed sales reports in a single collection called **sales**.

Follow the instructions below to load the supplies sales collection into MongoDB.

You will need to complete this step successfully to proceed to answer the questions in the component.

1). Start your MongoDB daemon, **mongod**.

2). Import the sales collection using the command invocation below:

**mongoimport --db=comp47470supplies --collection=sales --file=supplies.json**

On successful invocation of the command, you will get the following output:

```text
2024-02-29T20:04:55.062+0000	connected to: mongodb://localhost/
2024-02-29T20:04:55.482+0000	5000 document(s) imported successfully. 0 document(s) failed to import.
```

Launch the MongoDB shell, **mongosh**.

At the command prompt, type 'show databases'. You will see 'comp47470supplies' db listed.

```bash
test> show databases
admin                  40.00 KiB
comp47470supplies     968.00 KiB
config                 72.00 KiB
local                  72.00 KiB
test                  112.00 KiB
```

Switch to 'comp47470supplies' db and find which collections are available in the database.

```bash
test> use comp47470supplies
switched to db comp47470supplies
comp47470supplies> show collections
sales
```

Run the following queries:

1). Select one document from the collection **data** using the cursor **limit** directive as follows:

2). Find the number of documents. Use **countDocuments()** since it provides the exact count of documents.

```bash
comp47470supplies> db.sales.find( {} ).limit(1)
[
  {
    _id: ObjectId('5bd761dcae323e45a93ccfe8'),
    saleDate: ISODate('2015-03-23T21:06:49.506Z'),
    items: [
      {
        name: 'printer paper',
        tags: [ 'office', 'stationary' ],
        price: Decimal128('40.01'),
        quantity: 2
      },
...

comp47470supplies> db.sales.countDocuments( {} )
5000
```

If you are able to run this query successfully, then you are ready to tackle the following questions in the component.

In [ ]:
from pymongo import MongoClient
from bson import json_util


client = MongoClient("mongodb://localhost:27017/")


db = client["supplies"]
collection_sales = db["sales"]


print("Databases:", client.list_database_names())
print("Collections in supplies:", db.list_collection_names())


Databases: ['admin', 'comp47470financialdb', 'config', 'local', 'myTestDB', 'supplies']
Collections in supplies: ['sales']


In [ ]:
json_file_path = "supplies.json"


with open(json_file_path, 'r') as f:
    data = [json_util.loads(line) for line in f]

print(f"{len(data)} documents ready for processing.")


if collection_sales.count_documents({}) == 0:
    collection_sales.insert_many(data)
    print("Documents inserted into MongoDB.")
else:
    print("Collection already has data.")


5000 documents ready for processing.
Collection already has data.


In [ ]:

doc = collection_sales.find_one()
print(doc)


{'_id': ObjectId('5bd761dcae323e45a93ccfe8'), 'saleDate': datetime.datetime(2015, 3, 23, 21, 6, 49, 506000), 'items': [{'name': 'printer paper', 'tags': ['office', 'stationary'], 'price': Decimal128('40.01'), 'quantity': 2}, {'name': 'notepad', 'tags': ['office', 'writing', 'school'], 'price': Decimal128('35.29'), 'quantity': 2}, {'name': 'pens', 'tags': ['writing', 'office', 'school', 'stationary'], 'price': Decimal128('56.12'), 'quantity': 5}, {'name': 'backpack', 'tags': ['school', 'travel', 'kids'], 'price': Decimal128('77.71'), 'quantity': 2}, {'name': 'notepad', 'tags': ['office', 'writing', 'school'], 'price': Decimal128('18.47'), 'quantity': 2}, {'name': 'envelopes', 'tags': ['stationary', 'office', 'general'], 'price': Decimal128('19.95'), 'quantity': 8}, {'name': 'envelopes', 'tags': ['stationary', 'office', 'general'], 'price': Decimal128('8.08'), 'quantity': 3}, {'name': 'binder', 'tags': ['school', 'general', 'organization'], 'price': Decimal128('14.16'), 'quantity': 3}], 

## Problem 1

Write a MongoDB **read** query that outputs the number of documents satisfying **all** the following conditions:

1). The sale items include 'pens' and 'envelopes'.

2). The sale tags include 'office' and 'school'.

3). The customer is a female ('F').

4). The customer age must be greater than 60 (> 60).

5). The customer satisfaction must equal 5.

6). The purchase method is 'Online'.

7). The coupon used is true.

**What is the read query statement and the output from the statement execution?**

In [ ]:
query = {
    "items": {
        "$all": [
            {"$elemMatch": {"name": "pens", "tags": {"$all": ["office", "school"]}}},
            {"$elemMatch": {"name": "envelopes", "tags": {"$all": ["office", "school"]}}}
        ]
    },
    "customer.gender": "F",
    "customer.age": {"$gt": 60},
    "customer.satisfaction": 5,
    "purchaseMethod": "Online",
    "couponUsed": True
}


count = collection_sales.count_documents(query)
print("Number of matching documents:", count)



Number of matching documents: 0


## Problem 2

Write a MongoDB **update** statement that sets the coupon used to **false** for all the documents selected using a read query that satisfies **all** the conditions in **Problem 2**. Note the query must satisfy the following conditions:

1). The sale items include 'pens' and 'envelopes'.

2). The sale tags include 'office' and 'school'.

3). The customer is a female ('F').

4). The customer age must be greater than 60 (> 60).

5). The customer satisfaction must equal 5.

6). The purchase method is 'Online'.

7). The coupon used is **true**.

**What is the update statement and the output from the statement execution?**

In [ ]:

update_result = collection_sales.update_many(
    query,               
    {"$set": {"couponUsed": False}}  
)


print("Number of documents updated:", update_result.modified_count)


Number of documents updated: 0


## Problem 3

Write a MongoDB **delete** statement that deletes all the documents selected using a read query that satisfies **all** the following conditions:

1). The sale items include 'pens' and 'envelopes'.

2). The sale tags include 'office' and 'school'.

3). The customer is a female ('F').

4). The customer age must be greater than 60 (> 60).

5). The customer satisfaction must equal 5.

6). The purchase method is 'Online'.

7). The coupon used is **false**.

**What is the delete statement and the output from the statement execution?**

In [ ]:

delete_query = {
    "items": {
        "$all": [
            {"$elemMatch": {"name": "pens", "tags": {"$all": ["office", "school"]}}},
            {"$elemMatch": {"name": "envelopes", "tags": {"$all": ["office", "school"]}}}
        ]
    },
    "customer.gender": "F",
    "customer.age": {"$gt": 60},
    "customer.satisfaction": 5,
    "purchaseMethod": "Online",
    "couponUsed": False
}


delete_result = collection_sales.delete_many(delete_query)


print("Number of documents deleted:", delete_result.deleted_count)


Number of documents deleted: 0
